In [1]:
# Install OpenNMT-py 3.x
!pip3 install OpenNMT-py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.8/262.8 KB 13.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 KB 68.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 KB 42.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 148.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 4.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 35.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 167.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 187.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.7/110.7 KB 69.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.0/103.0 KB 65.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.2/29.2 MB 74.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.4/38.4 MB 60.9 MB/s eta 0:00:0000

In [7]:
# Create the YAML configuration file
# On a regular machine, you can create it manually or with nano
# Note here we are using some smaller values because the dataset is small
# For larger datasets, consider increasing: train_steps, valid_steps, warmup_steps, save_checkpoint_steps, keep_checkpoint

config = '''# config.yaml


## Where the samples will be written
save_data: run

# Training files
data:
    corpus_1:
        path_src: en-zh.en-filtered-salient.en.subword.train
        path_tgt: en-zh.zh-filtered.zh.subword.train
        transforms: [filtertoolong]
    valid:
        path_src: en-zh.en-filtered-salient.en.subword.dev
        path_tgt: en-zh.zh-filtered.zh.subword.dev
        transforms: [filtertoolong]

# Vocabulary files, generated by onmt_build_vocab
src_vocab: run/source.vocab
tgt_vocab: run/target.vocab

# Vocabulary size - should be the same as in sentence piece
src_vocab_size: 10000
tgt_vocab_size: 10000

# Filter out source/target longer than n if [filtertoolong] enabled
src_seq_length: 512
src_seq_length: 512

# Tokenization options
src_subword_model: source.model
tgt_subword_model: target.model

# Where to save the log file and the output models/checkpoints
log_file: train.log
save_model: models/model.base

# Stop training if it does not imporve after n validations
early_stopping: 4

# Default: 5000 - Save a model checkpoint for each n
save_checkpoint_steps: 2000

# To save space, limit checkpoints to last n
# keep_checkpoint: 3

seed: 3435

# Default: 100000 - Train the model to max n steps 
# Increase to 200000 or more for large datasets
# For fine-tuning, add up the required steps to the original steps
train_steps: 10000

# Default: 10000 - Run validation after n steps
valid_steps: 2000

# Default: 4000 - for large datasets, try up to 8000
warmup_steps: 4000
report_every: 100

# Number of GPUs, and IDs of GPUs
world_size: 1
gpu_ranks: [0]

# Batching
bucket_size: 262144
num_workers: 0  # Default: 2, set to 0 when RAM out of memory
batch_type: "tokens"
batch_size: 4096   # Tokens per batch, change when CUDA out of memory
valid_batch_size: 2048
max_generator_batches: 2
accum_count: [4]
accum_steps: [0]

# Optimization
model_dtype: "fp16"
optim: "adam"
learning_rate: 2
# warmup_steps: 8000
decay_method: "noam"
adam_beta2: 0.998
max_grad_norm: 0
label_smoothing: 0.1
param_init: 0
param_init_glorot: true
normalization: "tokens"
weight_decay: 0.0001

# Model
encoder_type: transformer
decoder_type: transformer
position_encoding: true
enc_layers: 6
dec_layers: 6
heads: 8
hidden_size: 512
word_vec_size: 512
transformer_ff: 2048
dropout_steps: [0]
dropout: [0.1]
attention_dropout: [0.1]
'''

with open("config.yaml", "w+") as config_yaml:
  config_yaml.write(config)

In [8]:
# Find the number of CPUs/cores on the machine
!nproc --all

64


In [21]:
# Build Vocabulary

# -config: path to your config.yaml file
# -n_sample: use -1 to build vocabulary on all the segment in the training dataset
# -num_threads: change it to match the number of CPUs to run it faster

!onmt_build_vocab -config config.yaml -n_sample -1 -num_threads 16


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/venv/main/bin/onmt_build_vocab", line 5, in <module>
    from onmt.bin.build_vocab import main
  File "/venv/main/lib/python3.10/site-packages/onmt/__init__.py", line 2, in <module>
    import onmt.inputters
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/__init__.py", line 7, in <module>
    from onmt.inputters.text_utils import text_sort_key, process, numericalize, tensorify
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/text_utils.py", line 1, in <module>
    import torch
  File "

In [5]:
# Check if the GPU is active
!nvidia-smi -L

GPU 0: NVIDIA L40S (UUID: GPU-91006845-7b0b-d094-209b-a3a40a8ccfc6)


In [6]:
# Check if the GPU is visable to PyTorch
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

gpu_memory = torch.cuda.mem_get_info(0)
print("Free GPU memory:", gpu_memory[0]/1024**2, "out of:", gpu_memory[1]/1024**2)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/venv/main/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/venv/main/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/venv/main/lib/python3.10/site-packages/ipykernel/kernelapp.p

True
NVIDIA L40S
Free GPU memory: 45158.25 out of: 45589.0625


In [22]:
# Train the NMT model
!onmt_train -config config.yaml


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/venv/main/bin/onmt_train", line 5, in <module>
    from onmt.bin.train import main
  File "/venv/main/lib/python3.10/site-packages/onmt/__init__.py", line 2, in <module>
    import onmt.inputters
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/__init__.py", line 7, in <module>
    from onmt.inputters.text_utils import text_sort_key, process, numericalize, tensorify
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/text_utils.py", line 1, in <module>
    import torch
  File "/venv/main/l

## Translate

In [23]:
# Translate the "subworded" source file of the test dataset
# Change the model name, if needed.
# gpu
!onmt_translate -model models/model.base_step_10000.pt -src en-zh.en-filtered-salient.en.subword.test -output zh.translated -gpu 0 -min_length 1




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/venv/main/bin/onmt_translate", line 5, in <module>
    from onmt.bin.translate import main
  File "/venv/main/lib/python3.10/site-packages/onmt/__init__.py", line 2, in <module>
    import onmt.inputters
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/__init__.py", line 7, in <module>
    from onmt.inputters.text_utils import text_sort_key, process, numericalize, tensorify
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/text_utils.py", line 1, in <module>
    import torch
  File "/ven

In [14]:
%pip install "numpy<2"


Note: you may need to restart the kernel to use updated packages.


In [24]:
# Check the first 5 lines of the translation file
!head -n 30 zh.translated

▁ 库 兹 洛 克 的 议员 们 , 东 德国 , ▁ 爱 沙 尼亚 、 拉 维 亚 、 里 维 亚 、 马 达 加 斯 加 、 马 达 加 斯 加 、 波 兰 、 ▁ 菲 尔 维 亚 、 塞 尔 维 亚 、 塞 尔 维 亚 、 斯 、 突 尼 斯 、 埃及 。
▁我不知道 他们 要 怎么 处理 这些
▁它 需要 一些 构 思 , ▁因为它 需要 一个 系统 , ▁而不是 雕塑 。
▁ 伟大的 教师 也 这样做 , ▁但 伟大的 教师 也在 做的就是 导师 , ▁ 刺激 , 激发 。
▁所以我 只是 穿 上 这 根 脏 布 , ▁ 全 是 4 0 0 0 个 试 点 , ▁当我 接近 失去 理智 的时候 , ▁我 找到了 蛋白质 。
▁我 注意到 他们 能 通过 一 度 或 二 度 ▁ 在家里 改变 体 温 。
▁但这 是 后果 。
▁那么 ,“ 第二 人生 ” 的 平均 年龄 是 3 2 岁 , ▁然而 ,“ 第二 人生 ” 的 使用 量 ▁ 随着 物理 年龄 增长 ▁ 剧 烈 地 增长 。 所以 当你 从 3 0 岁 到 6 0 岁 以上 —— ▁很多人 在 “ 第二 人生 ” 中 使用 “ 第二 人生 ” —— ▁ 这条 线 并不是 一个 明显 的 增长 —— ▁非常 分散 的 —— ▁比如 , 每周 4 0 小时 , 4 0 %
▁因此 , 它 持续 思考 着 它的 抽象 思维 。
▁ 这项技术 的 好处 在于 ▁ 可以让 手机 开始 看见 和 了解 ▁人类 大脑 所做的 事 。
▁由于 建筑师 的 帮助 , 居民 的 帮助 ▁就 从 地 上 提高 了 起来 。
▁然后 学生 会 走进 我们的 录音 室 , ▁他们会 用 自己的 节奏 ▁ 制作 自己的 说 唱 歌曲 。
▁有时 我 从 7 天 的 阿 凡 特 主义 教堂 ▁ 给我 看了 这些 卡通 画 。
▁如果你 用 小 农场 主 的 农业 来 填 满 这个 杯子 , ▁你 就有了 一个 转变 的作用 。
▁然后 , 她 开了 一 小时 , 问 :“ 你 是谁 ?”
▁现在 , 非常 清楚 的是 , 我将 一起 收集 那 副 牌 。
▁这是一个 统一 的 系统 , ▁尽管 总体 规划 和 发展 良好 。
▁然后 现代 人 出现 在非洲 的 某个 地方 , ▁ 出生 于 非洲 , 大概 在 中东 。
▁这些

In [25]:
# If needed install/update sentencepiece
!pip3 install --upgrade -q sentencepiece

# Desubword the translation file
!python3 ./MT-Preparation/subwording/3-desubword.py ./target.model zh.translated

Done desubwording! Output: zh.translated.desubword


In [26]:
# Desubword the target file (reference) of the test dataset
# Note: You might as well have split files *before* subwording during dataset preperation, 
# but sometimes datasets have tokeniztion issues, so this way you are sure the file is really untokenized.
!python3 ./MT-Preparation/subwording/3-desubword.py ./source.model en-zh.en-filtered-salient.en.subword.test

# Desubword the test file
!python3 ./MT-Preparation/subwording/3-desubword.py ./target.model en-zh.zh-filtered.zh.subword.test

Done desubwording! Output: en-zh.en-filtered-salient.en.subword.test.desubword
Done desubwording! Output: en-zh.zh-filtered.zh.subword.test.desubword


In [27]:
# Check the first 5 lines of the desubworded translation file
!head -n 30 zh.translated.desubword

库兹洛克的议员们,东德国, 爱沙尼亚、拉维亚、里维亚、马达加斯加、马达加斯加、波兰、 菲尔维亚、塞尔维亚、塞尔维亚、斯、突尼斯、埃及。
我不知道他们要怎么处理这些
它需要一些构思, 因为它需要一个系统, 而不是雕塑。
伟大的教师也这样做, 但伟大的教师也在做的就是导师, 刺激,激发。
所以我只是穿上这根脏布, 全是4000个试点, 当我接近失去理智的时候, 我找到了蛋白质。
我注意到他们能通过一度或二度 在家里改变体温。
但这是后果。
那么,“第二人生”的平均年龄是32岁, 然而,“第二人生”的使用量 随着物理年龄增长 剧烈地增长。所以当你从30岁到60岁以上—— 很多人在“第二人生”中使用“第二人生”—— 这条线并不是一个明显的增长—— 非常分散的—— 比如,每周40小时,40%
因此,它持续思考着它的抽象思维。
这项技术的好处在于 可以让手机开始看见和了解 人类大脑所做的事。
由于建筑师的帮助,居民的帮助 就从地上提高了起来。
然后学生会走进我们的录音室, 他们会用自己的节奏 制作自己的说唱歌曲。
有时我从7天的阿凡特主义教堂 给我看了这些卡通画。
如果你用小农场主的农业来填满这个杯子, 你就有了一个转变的作用。
然后,她开了一小时,问:“你是谁?”
现在,非常清楚的是,我将一起收集那副牌。
这是一个统一的系统, 尽管总体规划和发展良好。
然后现代人出现在非洲的某个地方, 出生于非洲,大概在中东。
这些失调可以帮助我们解决问题, 它们帮助我们变得更有创造力。
格伦看起来老一点。
如果可以的话,那不是很棒吗? 头一次,你可以将你的眼内衣 完美地融入你 并且不需要任何的装配, 所以机会是,  ⁇ 不会碎?
现在,中国的经济发展 则是一个戏剧化, 基本的改变。 25年前,也就是发展中国家, 全世界的贫穷国家, 都不是相当大, 他们占据了全世界总数的三分之一。
当然,现在在家中我非常敏感, 当我们把灯关掉的时候。
表现最差的国家,中非共和国,31分。
这些都是人类本能的冲动, 但因为技术, 这些冲动只是一声罢了。
这是我第一次看到的东西, 当我第一次跳入水中 跳入西班牙的海岸。
现在,这在世界杯竞赛中起了重要作用。
我们确实找到了
我认识的消防员告诉我,这不是一个分类。
不,我想他在讲选择森林


## Evaluation

In [28]:
# Download the BLEU script
!wget https://raw.githubusercontent.com/ymoslem/MT-Evaluation/main/BLEU/compute-bleu.py

--2025-04-12 13:04:30--  https://raw.githubusercontent.com/ymoslem/MT-Evaluation/main/BLEU/compute-bleu.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
200 OKequest sent, awaiting response... 
Length: 957 [text/plain]
Saving to: ‘compute-bleu.py.3’

compute-bleu.py.3   100%[===================>]     957  --.-KB/s    in 0s      

2025-04-12 13:04:30 (53.1 MB/s) - ‘compute-bleu.py.3’ saved [957/957]



In [29]:
# Install sacrebleu
!pip3 install sacrebleu

In [30]:
# Evaluate the translation (without subwording)
!python3 compute-bleu.py en-zh.zh-filtered.zh.subword.test.desubword zh.translated.desubword

Reference 1st sentence: 在捷克斯洛伐克,东德 爱沙尼亚,拉脱维亚,立陶宛, 马里,马达加斯加, 波兰,菲律宾, 塞尔维亚,斯洛维尼亚的独裁政府,我可以继续, 还有现在的突尼斯和埃及。
MTed 1st sentence: 库兹洛克的议员们,东德国, 爱沙尼亚、拉维亚、里维亚、马达加斯加、马达加斯加、波兰、 菲尔维亚、塞尔维亚、塞尔维亚、斯、突尼斯、埃及。
BLEU:  3.37646848432582
